# Uncertainty Quantification: Ensembles and Latent-Gaussian Forecasts

This notebook demonstrates predictive intervals for `GraphKoopmanModel` via
the power-user module `koopman_graph.uq`.

## What this is (and is not)

- **Deep ensembles** (Lakshminarayanan et al., NeurIPS 2017): train several
  independently seeded models and form an empirical mean plus quantile
  bounds with `predict_interval`. Epistemic diversity comes from different
  initializations / fits.
- **Latent-Gaussian / Kalman-refined UQ:** propagate a Gaussian latent under
  \(z\mapsto Kz\) with closed-form covariance \(P\leftarrow APA^\top+Q\), then
  form observation-space intervals by Monte Carlo decoding. Optional
  mid-horizon observations apply a Kalman update (related to the Kalman half
  of K²VAE-style pipelines; **not** a full VAE + KalmanNet reimplementation).
- **Not Deep Probabilistic Koopman (DPK):** DPK typically predicts
  time-varying distribution parameters (often without the same step-by-step
  latent rollout contract used here).

Both wrappers **compose** ordinary `GraphKoopmanModel` instances; they do not
subclass the primary model or fork latent rollout. Import from
`koopman_graph.uq` (not the root façade).

In [ ]:
import torch
import matplotlib.pyplot as plt

from koopman_graph import GNNDecoder, GNNEncoder, GraphKoopmanModel
from koopman_graph.datasets import SyntheticDynamicGraphBenchmark
from koopman_graph.uq import (
    EnsembleGraphKoopmanModel,
    LatentGaussianKoopmanUQ,
    empirical_coverage,
)

## Noisy synthetic diffusion

Generate a small Laplacian-diffusion sequence with additive noise, then hold
out a short open-loop forecast window for coverage checks.

In [ ]:
sequence = SyntheticDynamicGraphBenchmark.generate(
    num_nodes=8,
    num_timesteps=40,
    in_channels=1,
    topology="ring",
    noise_std=0.04,
    seed=42,
)
train = sequence[:28]
initial = train[-1]
targets = list(sequence[28:34])
print(f"train steps={len(train)}, forecast horizon={len(targets)}")

## Fit an independently seeded ensemble

`from_factory` builds members under distinct RNG seeds. Optional `seeds=` on
`fit` reseeds before each member's training loop.

In [ ]:
def make_member() -> GraphKoopmanModel:
    encoder = GNNEncoder(in_channels=1, hidden_channels=8, latent_dim=4)
    decoder = GNNDecoder(latent_dim=4, hidden_channels=8, out_channels=1)
    return GraphKoopmanModel(
        encoder=encoder,
        decoder=decoder,
        latent_dim=4,
        time_step=1.0,
    )

ensemble = EnsembleGraphKoopmanModel.from_factory(
    make_member,
    n_members=5,
    seeds=(0, 1, 2, 3, 4),
)
histories = ensemble.fit(
    train,
    epochs=40,
    lr=1e-2,
    seeds=(100, 101, 102, 103, 104),
)
print("final train losses:", [round(h.loss[-1], 4) for h in histories])

## Predictive intervals and empirical coverage

`predict_interval(..., level=0.9)` returns the ensemble mean plus empirical
5th/95th percentiles across members. `empirical_coverage` measures the
fraction of held-out entries inside those bounds.

In [ ]:
level = 0.9
interval = ensemble.predict_interval(initial, steps=len(targets), level=level)
coverage = empirical_coverage(targets, interval)
print(f"nominal level={level:.2f}, empirical coverage={coverage:.3f}")

node = 0
steps = range(1, len(targets) + 1)
truth = [snap.x[node, 0].item() for snap in targets]
mean = [snap.x[node, 0].item() for snap in interval.mean]
lower = [snap.x[node, 0].item() for snap in interval.lower]
upper = [snap.x[node, 0].item() for snap in interval.upper]

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.fill_between(steps, lower, upper, alpha=0.25, label=f"{int(level*100)}% interval")
ax.plot(steps, mean, marker="o", label="ensemble mean")
ax.plot(steps, truth, marker="x", linestyle="--", label="truth")
ax.set_xlabel("forecast step")
ax.set_ylabel(f"node {node} feature")
ax.set_title("Deep-ensemble predictive interval")
ax.legend()
fig.tight_layout()
plt.show()

## Comparison: latent-Gaussian forecast UQ

Fit a single model and wrap it with `LatentGaussianKoopmanUQ`. Predictive
intervals come from Monte Carlo samples of the propagated latent Gaussian —
honest under a nonlinear GNN decoder, and distinct from ensemble epistemic
diversity.

In [ ]:
# Reuse a fitted ensemble member (no second training loop).
gauss = LatentGaussianKoopmanUQ(
    ensemble.members[0],
    process_noise=1e-3,
    initial_covariance=1.0,
    n_samples=64,
)
gauss_interval = gauss.predict_interval(initial, steps=len(targets), level=level)
gauss_coverage = empirical_coverage(targets, gauss_interval)
print(
    f"ensemble coverage={coverage:.3f}, "
    f"latent-Gaussian coverage={gauss_coverage:.3f}"
)

g_mean = [snap.x[node, 0].item() for snap in gauss_interval.mean]
g_lower = [snap.x[node, 0].item() for snap in gauss_interval.lower]
g_upper = [snap.x[node, 0].item() for snap in gauss_interval.upper]

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, title, m, lo, hi, cov in [
    (axes[0], "Ensemble", mean, lower, upper, coverage),
    (axes[1], "Latent-Gaussian", g_mean, g_lower, g_upper, gauss_coverage),
]:
    ax.fill_between(steps, lo, hi, alpha=0.25, label=f"{int(level*100)}% interval")
    ax.plot(steps, m, marker="o", label="mean")
    ax.plot(steps, truth, marker="x", linestyle="--", label="truth")
    ax.set_title(f"{title} (cov={cov:.2f})")
    ax.set_xlabel("forecast step")
    ax.legend(fontsize=8)
axes[0].set_ylabel(f"node {node} feature")
fig.tight_layout()
plt.show()

# Soft teaching check: both paths should produce finite coverage in (0, 1].
assert 0.0 < coverage <= 1.0
assert 0.0 < gauss_coverage <= 1.0
assert len(gauss_interval.mean) == len(targets)

## Takeaways

- Use `EnsembleGraphKoopmanModel` for epistemic uncertainty via independently
  seeded fits; use `LatentGaussianKoopmanUQ` for linear-Gaussian latent
  propagation (optional Kalman refine) on a **single** fitted model.
- Both expose optional `predict_interval` (`IntervalForecastModel`); neither
  is required on `ForecastModel`, and neither is on the root façade.
- Name claims carefully: ensembles ≠ DPK; latent-Gaussian UQ ≠ full K²VAE and
  ≠ DPK.
- Coverage on a short noisy teaching example is illustrative; calibrate on
  your domain before treating intervals as decision thresholds.